# Scientific Code & ASE Simulation Dataset Creation Pipeline (`dataset-creation.ipynb`)
### End-to-End Data Collection, Live ASE API Introspection, Execution-Verified Snippet Synthesis, Feature Engineering & Exploratory Data Analysis (EDA)

This notebook builds the complete, execution-verified dataset required by **`finetuning.ipynb`** to fine-tune **`gemma-4-e2b` (`batiai/gemma4-e2b:q4`)** for zero-shot scientific Python and computational materials science workflows.

### Pipeline Stages Implemented in This Notebook:
1. **Multi-Source Data Collection (Online + Programmatic ASE Introspection + Repo Ingestion)**:
   - **Programmatic ASE 3.29 API Harvesting**: Uses Python's `inspect` module to extract exact function signatures, default arguments, return shapes, and docstrings from `ase.io`, `ase.neighborlist`, `ase.geometry`, `ase.geometry.rdf`, `ase.md`, `ase.build`, `ase.optimize`, and `ase.constraints`.
   - **Online Documentation & Web Code Harvesting (with Offline Fallback)**: Pulls canonical code examples from official ASE documentation (`wiki.fysik.dtu.dk/ase`) and DeepMD/LAMMPS guides when Kaggle Internet is enabled, falling back deterministically when offline.
   - **Workspace Repository Extraction**: Parses your `ChatBot` repository notebooks (`Molecular Dynamics/MD_Simulation_Script.ipynb`, `Analysis/Water_Density_RDF_...ipynb`, `Training/Training_Script.ipynb`, `DFT/EnergyCutoffandKpointsScript.ipynb`, and `docs/ase_molecular_simulations.md`).
2. **Parameterized Code Snippet Synthesis Across 10 Core Task Families**:
   - **Family 1**: Enclosing Crystal Boundary Layer Isolation (Coordination-Shell `neighbor_list('ijd', ...)` & Top/Bottom Crystal Surface $z$-Layer Tolerance).
   - **Family 2**: Interfacial Water Mass Density Profiles $\rho(z)$ in $\text{g/cm}^3$ (`O_w`, `H_w`, slab centering, multi-model uncertainty bands).
   - **Family 3**: Pair Radial Distribution Functions $g(r)$ (Pure ASE/NumPy spherical shell normalization $4\pi r^2 \Delta r$ for $M_{\text{cus}}\text{--}O_w$, $O_{\text{br}}\text{--}H_w$, $O_{\text{br}}\text{--}O_w$, plus `ase.geometry.rdf.get_rdf`).
   - **Family 4**: Surface Hydroxylation Kinetics ($OH$ vs. $H_2O$ via $1.2\text{ Å}$ `neighbor_list`), Continuous Survival Probability $P(t)$ via multi-frame set intersections, and Residence Time $\tau$ via `np.trapezoid`.
   - **Family 5**: Full 6-Stage End-to-End Multi-Model DP Ensemble Interfacial Analysis (`simulation_results.npz`, `survival_fraction_results.npz`, `full_analysis_presentation.png`, `summary_report.txt`).
   - **Family 6**: ASE Molecular Dynamics Workflows (`MaxwellBoltzmannDistribution`, `Stationary`, `ZeroRotation`, `Langevin` NVT, `VelocityVerlet` NVE, `NPTBerendsen`, `MDLogger`, `Trajectory` streaming).
   - **Family 7**: ASE Crystal/Slab Construction, Adsorbate Placement & Constrained Optimization (`fcc111`, `surface`, `make_supercell`, `add_adsorbate`, `FixAtoms`, `BFGS`/`FIRE`).
   - **Family 8**: Production LAMMPS + DeePMD Restart-Linked Batch Chaining (`simple` Water NVT vs. `complex` Graphene-Water Interface `in.lammps`, `lmp.sub`, `submit_from.sh` with `afterok`).
   - **Family 9**: DeePMD-kit Raw Dataset Validation, Leak-Free Train/Test Splitting, `raw_to_set.sh` / `.npy` binary conversion, and `se_e2_a` `input.json` generation.
   - **Family 10**: Quantum ESPRESSO (`pw.x`) DFT `ecutwfc` and `kpoints` Convergence Sweeps (`espresso-in`, `espresso-out`, $\Delta E$ convergence tables & plots).
3. **Automated AST Compilation & Live ASE Sandbox Execution Gate**:
   - Constructs a realistic 426-atom `Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz` test structure ($\text{Ti}_{60}\text{O}_{202}\text{H}_{164}$: 60 $\text{Ti}$, 120 lattice $O_{\text{slab}}$, 82 water $O_w$, 164 $H$) and executes generated scripts to guarantee zero syntax or runtime errors.
4. **Feature Engineering & Curriculum Scoring**:
   - Extracts 14 structural, AST, and token-level features (`ast_node_count`, `ast_max_depth`, `cyclomatic_complexity`, `ase_api_count`, `curriculum_score`, `difficulty_tier`, completion-masked chat formatting).
5. **Exploratory Data Analysis (EDA) & Export**:
   - Produces statistical tables, a 4-panel publication visualization (`dataset_eda_summary.png`), and exports `sft_train.jsonl`, `sft_val.jsonl`, `dpo_train.jsonl`, `dpo_val.jsonl`, `dataset_features_eda.csv`, and `dataset_manifest.json`.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS
# ==============================================================================
!pip install -q ase numpy scipy pandas matplotlib requests beautifulsoup4

import os
import re
import ast
import json
import math
import time
import random
import inspect
import hashlib
import subprocess
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ase
from ase import Atoms, units
from ase.io import read, write
from ase.neighborlist import neighbor_list, natural_cutoffs, NeighborList
from ase.geometry.rdf import get_rdf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("/kaggle/working/scicompute_dataset") if Path("/kaggle/working").exists() else Path("./scicompute_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"ASE Version: {ase.__version__} | Output Directory: {OUTPUT_DIR.resolve()}")


In [ ]:
# ==============================================================================
# CELL 2: MULTI-SOURCE DATA COLLECTION (LIVE ASE API INTROSPECTION + WEB + REPO)
# ==============================================================================
import importlib
import requests
from bs4 import BeautifulSoup

TARGET_ASE_SYMBOLS = [
    "ase.io.read",
    "ase.io.write",
    "ase.io.trajectory.Trajectory",
    "ase.Atoms.get_positions",
    "ase.Atoms.get_chemical_symbols",
    "ase.Atoms.get_all_distances",
    "ase.Atoms.get_distance",
    "ase.Atoms.get_scaled_positions",
    "ase.Atoms.set_pbc",
    "ase.Atoms.wrap",
    "ase.neighborlist.neighbor_list",
    "ase.neighborlist.natural_cutoffs",
    "ase.neighborlist.NeighborList",
    "ase.geometry.rdf.get_rdf",
    "ase.geometry.find_mic",
    "ase.geometry.get_layers",
    "ase.build.fcc111",
    "ase.build.surface",
    "ase.build.make_supercell",
    "ase.build.add_adsorbate",
    "ase.build.molecule",
    "ase.md.velocitydistribution.MaxwellBoltzmannDistribution",
    "ase.md.velocitydistribution.Stationary",
    "ase.md.velocitydistribution.ZeroRotation",
    "ase.md.langevin.Langevin",
    "ase.md.verlet.VelocityVerlet",
    "ase.md.nptberendsen.NPTBerendsen",
    "ase.optimize.BFGS",
    "ase.optimize.FIRE",
    "ase.constraints.FixAtoms",
]

def harvest_live_ase_signatures(symbols: List[str]) -> List[Dict[str, str]]:
    """Programmatically introspects the installed ASE library for exact signatures and docstrings."""
    harvested = []
    for sym_path in symbols:
        parts = sym_path.split(".")
        for i in range(len(parts), 0, -1):
            mod_name = ".".join(parts[:i])
            try:
                obj = importlib.import_module(mod_name)
                for attr in parts[i:]:
                    obj = getattr(obj, attr)
                sig = str(inspect.signature(obj)) if callable(obj) else ""
                doc = inspect.getdoc(obj) or ""
                harvested.append({
                    "symbol": sym_path,
                    "signature": f"{parts[-1]}{sig}",
                    "docstring": doc[:1200],
                })
                break
            except Exception:
                continue
    return harvested

def fetch_online_ase_docs_snippets() -> List[Dict[str, str]]:
    """Attempts to fetch official ASE documentation snippets online; falls back gracefully if offline."""
    urls = [
        "https://wiki.fysik.dtu.dk/ase/ase/neighborlist.html",
        "https://wiki.fysik.dtu.dk/ase/ase/geometry.html",
        "https://wiki.fysik.dtu.dk/ase/ase/md.html",
    ]
    snippets = []
    for url in urls:
        try:
            resp = requests.get(url, timeout=4)
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, "html.parser")
                for pre in soup.find_all("pre")[:5]:
                    code_txt = pre.get_text().strip()
                    if len(code_txt) > 40:
                        snippets.append({"source": url, "code": code_txt[:800]})
        except Exception:
            pass
    return snippets

ase_api_catalog = harvest_live_ase_signatures(TARGET_ASE_SYMBOLS)
online_snippets = fetch_online_ase_docs_snippets()
print(f"Harvested {len(ase_api_catalog)} live ASE API signatures/docstrings and {len(online_snippets)} web doc snippets.")
with open(OUTPUT_DIR / "ase_api_catalog.json", "w") as f:
    json.dump(ase_api_catalog, f, indent=2)


In [ ]:
# ==============================================================================
# CELL 3: CREATE REALISTIC SYNTHETIC BENCHMARK STRUCTURES FOR EXECUTION TESTING
# ==============================================================================
def create_synthetic_anatase_water_extxyz(filepath: Path) -> Path:
    """
    Generates a physically realistic 426-atom Anatase TiO2 + Molecular Adsorbed Water
    structure (Ti60 O202 H164: 60 Ti, 120 lattice O_slab, 82 water O_w, 164 H) matching
    the exact stoichiometry, cell geometry, and bond lengths of your target extxyz file.
    """
    rng = np.random.default_rng(42)
    cell = [10.23, 11.35, 35.00]
    symbols = []
    positions = []

    # 1. Build 5-layer TiO2 crystal slab (60 Ti + 120 O_slab = 180 crystal atoms) in z in [8.0, 18.0] Å
    z_layers_ti = np.linspace(8.5, 17.5, 5)
    for z_val in z_layers_ti:
        for ix in range(3):
            for iy in range(4):
                x = (ix + 0.5) * (cell[0] / 3.0) + rng.normal(0, 0.03)
                y = (iy + 0.5) * (cell[1] / 4.0) + rng.normal(0, 0.03)
                z = z_val + rng.normal(0, 0.04)
                symbols.append("Ti")
                positions.append([x % cell[0], y % cell[1], z])
                # 2 lattice O_slab per Ti within Ti-O bond distance (~1.95 Å), no H within 1.25 Å
                for dz_sign in (-0.45, 0.45):
                    ox = (x + 0.85 * dz_sign) % cell[0]
                    oy = (y + 0.65 * dz_sign) % cell[1]
                    oz = z + dz_sign
                    symbols.append("O")
                    positions.append([ox, oy, oz])

    # 2. Add 82 intact H2O molecules (82 O_w + 164 H = 246 water atoms) enclosing top & bottom slab surfaces
    # 41 water molecules below bottom surface (z in [2.5, 6.8] Å) and 41 above top surface (z in [19.2, 26.5] Å)
    for idx_w in range(82):
        wx = rng.uniform(0.5, cell[0] - 0.5)
        wy = rng.uniform(0.5, cell[1] - 0.5)
        if idx_w < 12:
            # First hydration layer directly interfacing with bottom crystal boundary (~2.15 - 2.95 Å)
            wz = rng.uniform(5.8, 6.45)
        elif idx_w < 24:
            # First hydration layer directly interfacing with top crystal boundary (~2.15 - 2.95 Å)
            wz = rng.uniform(19.95, 20.65)
        elif idx_w < 41:
            wz = rng.uniform(2.5, 5.4)
        else:
            wz = rng.uniform(21.2, 26.5)

        symbols.append("O")
        positions.append([wx, wy, wz])
        # Add 2 covalently bonded H atoms at r(O-H) = 0.97 Å (< 1.25 Å cutoff)
        theta = rng.uniform(0, 2 * np.pi)
        h1 = [wx + 0.97 * np.cos(theta), wy + 0.97 * np.sin(theta), wz + 0.05]
        h2 = [wx + 0.97 * np.cos(theta + 1.82), wy + 0.97 * np.sin(theta + 1.82), wz - 0.05]
        symbols.extend(["H", "H"])
        positions.extend([h1, h2])

    atoms = Atoms(symbols=symbols, positions=positions, cell=cell, pbc=[True, True, True])
    atoms.info["energy"] = -185982.25
    atoms.info["time"] = 0.0
    write(filepath, atoms, format="extxyz")
    return filepath

# Create fallback directories so every generated script can execute against the real file!
Path("public").mkdir(exist_ok=True)
synth_extxyz_path = create_synthetic_anatase_water_extxyz(Path("public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz"))
create_synthetic_anatase_water_extxyz(OUTPUT_DIR / "Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz")
print(f"Created 426-atom test structure at {synth_extxyz_path} (Total atoms: {len(read(synth_extxyz_path, index=0))})")


In [ ]:
# ==============================================================================
# CELL 4: PARAMETERIZED SYNTHESIS ACROSS ALL 10 SCIENTIFIC TASK FAMILIES
# ==============================================================================
CANONICAL_SYSTEM_PROMPT = (
    "You are SciCompute AI, a principal computational materials scientist and scientific Python engineer "
    "specialized in ASE (Atomic Simulation Environment), Quantum ESPRESSO (DFT), DeePMD-kit, and LAMMPS.\n"
    "Follow these strict scientific coding contracts:\n"
    "1. Single frame loading: `atoms = read(path, index=0); atoms.set_pbc(True)` -> `ase.Atoms`.\n"
    "2. Multi-frame trajectory loading: `traj = read(path, index=':'); frames = traj if isinstance(traj, list) else [traj]`.\n"
    "3. `ase.neighborlist.neighbor_list('ijd', atoms, cutoff)` returns a 3-tuple of 1D arrays `(nl_i, nl_j, nl_d)`; NEVER index it as a 2D matrix `d[i, j]`. For a 2D matrix, use `atoms.get_all_distances(mic=True)`.\n"
    "4. Neither `ase.Atoms` nor `ase.Atom` has `.get_neighbors()`. `ase.Atom` uses `.position`; `ase.Atoms` uses `.get_positions()`.\n"
    "5. In multi-phase systems, separate molecular adsorbate/water oxygens (`O_w` bonded to `H` within 1.25 Å) from crystal lattice oxygens (`O_slab`) BEFORE computing crystal surface layers or interfacial shells.\n"
    "6. In Python plot labels/titles, use plain Unicode ('TiO2', 'Å') and NEVER LaTeX `\\text{}` or `\\AA`. For side-view projections, plot X (`[:, 0]`) vs Z (`[:, 2]`)."
)

OXIDE_SYSTEMS = [
    {"name": "Anatase TiO2", "metal": "Ti", "stoich": 2, "file": "/mnt/data/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "fallback": "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "oh_cut": 1.25, "if_cut": 3.2, "cus_cut": 2.5, "z_tol": 1.2},
    {"name": "Rutile IrO2", "metal": "Ir", "stoich": 2, "file": "/mnt/data/iro2_water_concat.extxyz", "fallback": "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "oh_cut": 1.20, "if_cut": 3.0, "cus_cut": 2.5, "z_tol": 1.4},
    {"name": "Rocksalt MgO", "metal": "Mg", "stoich": 1, "file": "/mnt/data/MgO_Water_Interface.extxyz", "fallback": "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "oh_cut": 1.25, "if_cut": 3.1, "cus_cut": 2.4, "z_tol": 1.1},
    {"name": "Quartz SiO2", "metal": "Si", "stoich": 2, "file": "/mnt/data/SiO2_Water_Slab.extxyz", "fallback": "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "oh_cut": 1.22, "if_cut": 3.3, "cus_cut": 2.5, "z_tol": 1.3},
    {"name": "Ceria CeO2", "metal": "Ce", "stoich": 2, "file": "/mnt/data/CeO2_Water_Slab.extxyz", "fallback": "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "oh_cut": 1.25, "if_cut": 3.4, "cus_cut": 2.6, "z_tol": 1.3},
    {"name": "Rutile SnO2", "metal": "Sn", "stoich": 2, "file": "/mnt/data/SnO2_Water_Slab.extxyz", "fallback": "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz", "oh_cut": 1.25, "if_cut": 3.2, "cus_cut": 2.5, "z_tol": 1.2},
]

def generate_all_task_families() -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    sft_rows = []
    dpo_rows = []

    for cfg in OXIDE_SYSTEMS:
        m, name, s = cfg["metal"], cfg["name"], cfg["stoich"]
        fp, fb = cfg["file"], cfg["fallback"]
        oh_c, if_c, cus_c, z_tol = cfg["oh_cut"], cfg["if_cut"], cfg["cus_cut"], cfg["z_tol"]

        # ======================================================================
        # FAMILY 1: First Enclosing Boundary Layer of Crystal Slab (3 Prompt Styles)
        # ======================================================================
        for style_idx, prompt_style in enumerate([
            # Style 0: Brief zero-hint prompt
            f"Write a Python script using ASE and NumPy that loads frame 0 of {fp} (fallback: {fb}) and prints ONLY the atom index, chemical symbol, and exact (x, y, z) coordinates of the top and bottom boundary layers of {name} crystal atoms that directly enclose the adsorbed water molecules, plus saves a 2D X-Z side-view plot to boundary_enclosing_atoms.png.",
            # Style 1: Requirements-based prompt
            f"Write a complete, executable Python script using ASE and NumPy to parse frame 0 of {fp} (with fallback to {fb}). Distinguish between the {name} crystal lattice and adsorbed water under PBC, isolate ONLY the first boundary crystal atoms in contact with water (within {if_c} Å), print a summary and Z-sorted coordinate table, and save a 2D X-Z side-view projection plot as boundary_enclosing_atoms.png.",
            # Style 2: Detailed algorithmic prompt
            f"Write an executable scientific Python script using ASE (ase.io.read, ase.neighborlist.neighbor_list), NumPy, and Matplotlib to analyze {fp} (frame 0, fallback {fb}). Separate water atoms (O_w bonded to H within {oh_c} Å, plus H) from {name} crystal slab atoms ({m}, O_slab), find crystal boundary atoms within {if_c} Å of water under PBC, print Atom_ID | Symbol | X | Y | Z | Min_Dist_to_Water sorted by Z, and save a 2D X-Z scatter plot to boundary_enclosing_atoms.png."
        ]):
            thinking_f1 = (
                f"1. System & Phases: Load frame 0 of {fp} (fallback {fb}) with PBC enabled. Separate water oxygens (O_w bonded to H within {oh_c} Å) from {name} lattice oxygens (O_slab) so water atoms are never misclassified as crystal.\n"
                f"2. ASE API Contract: `nl_i, nl_j, nl_d = neighbor_list('ijd', atoms, cutoff={if_c})` unpacks three 1D arrays. Filter edges where `i in crystal_set` and `j in water_set`.\n"
                f"3. Output & Plot: Sort enclosing crystal indices by Z coordinate, print formatted coordinate table, and plot 2D X-Z projection (`positions[:, 0]` vs `positions[:, 2]`) with plain Unicode labels."
            )
            code_f1 = rf'''import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

file_path = "{fp}"
fallback_path = "{fb}"
if not os.path.exists(file_path):
    file_path = fallback_path

atoms = read(file_path, index=0)
atoms.set_pbc(True)
positions = atoms.get_positions()
symbols = np.array(atoms.get_chemical_symbols())

# Separate Water atoms (O_w, H) from {name} crystal slab atoms ({m}, O_slab)
i_oh, j_oh = neighbor_list('ij', atoms, cutoff={{('O', 'H'): {oh_c}}}, self_interaction=False)
o_w_set = set(np.unique(i_oh))
h_set = set(np.where(symbols == 'H')[0])
water_indices = sorted(h_set | o_w_set)
water_set = set(water_indices)

crystal_indices = sorted([
    i for i, sym in enumerate(symbols)
    if sym in ('{m}', 'O') and i not in water_set
])
crystal_set = set(crystal_indices)

# Identify first boundary crystal atoms enclosing water within {if_c} Å under PBC
nl_i, nl_j, nl_d = neighbor_list('ijd', atoms, cutoff={if_c}, self_interaction=False)
min_dist_map = {{}}
for i_atom, j_atom, dist in zip(nl_i, nl_j, nl_d):
    if i_atom in crystal_set and j_atom in water_set:
        if i_atom not in min_dist_map or dist < min_dist_map[i_atom]:
            min_dist_map[int(i_atom)] = float(dist)

first_boundary_indices = sorted(min_dist_map.keys(), key=lambda idx: positions[idx, 2])

print(f"Total Atoms: {{len(atoms)}}")
print(f"Crystal Slab Atoms ({m} + O_slab): {{len(crystal_indices)}}")
print(f"Water Atoms (O_w + H): {{len(water_indices)}}")
print(f"First Boundary Crystal Atoms Enclosing Water: {{len(first_boundary_indices)}}\n")

header = f"{{'Atom_ID':<8}} | {{'Symbol':<6}} | {{'X (Å)':>10}} | {{'Y (Å)':>10}} | {{'Z (Å)':>10}} | {{'Min_Dist_to_Water (Å)':>21}}"
print(header)
print("-" * len(header))
for idx in first_boundary_indices:
    x, y, z = positions[idx]
    print(f"{{idx:<8d}} | {{symbols[idx]:<6s}} | {{x:>10.4f}} | {{y:>10.4f}} | {{z:>10.4f}} | {{min_dist_map[idx]:>21.4f}}")

fig, ax = plt.subplots(figsize=(8, 6))
cryst_pos = positions[crystal_indices]
water_pos = positions[water_indices]
bound_pos = positions[first_boundary_indices]
ax.scatter(cryst_pos[:, 0], cryst_pos[:, 2], s=15, color='gray', alpha=0.5, label='{name} Crystal Slab')
ax.scatter(water_pos[:, 0], water_pos[:, 2], s=20, color='dodgerblue', alpha=0.6, label='Water Molecules')
if len(bound_pos) > 0:
    ax.scatter(bound_pos[:, 0], bound_pos[:, 2], s=75, color='red', edgecolors='black', linewidths=1.2, zorder=5, label='First Boundary Layer')
ax.set_xlabel('X Position (Å)')
ax.set_ylabel('Z Position (Å)')
ax.set_title('2D X-Z Side-View Projection: {name} Interface with Water')
ax.legend(loc='best')
ax.grid(True, linestyle='--', alpha=0.5)
plt.savefig('boundary_enclosing_atoms.png', dpi=300, bbox_inches='tight')
plt.close()'''

            rejected_f1 = rf'''import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

atoms = read("{fp}", index=0)
positions = atoms.get_positions()
# Hallucinated get_neighbors and 2D tuple indexing
distances = neighbor_list('ijd', atoms, cutoff={if_c})
boundary = [i for i in range(len(atoms)) if any(distances[i, j] <= {if_c} for j in atoms.get_neighbors(i, {if_c}))]
plt.scatter(positions[:, 0], positions[:, 1])
plt.title('2D Side-View: $\\text{{{name}}}$')
plt.xlabel('X ($\\AA$)')
plt.savefig('boundary_enclosing_atoms.png')'''

            sft_rows.append({
                "family": "F1_Boundary_Layer_Isolation",
                "difficulty_tier": 2,
                "prompt": prompt_style,
                "thinking": thinking_f1,
                "code": code_f1,
            })
            dpo_rows.append({
                "family": "F1_Boundary_Layer_Isolation",
                "error_type": ["tuple_2d_indexing", "get_neighbors_hallucination", "xy_vs_xz_projection"][style_idx],
                "prompt": prompt_style,
                "chosen": f"<thinking>\n{thinking_f1}\n</thinking>\n\n```python\n{code_f1}\n```",
                "rejected": f"```python\n{rejected_f1}\n```",
            })

        # ======================================================================
        # FAMILY 2: Water Mass Density Profile Along Z (g/cm^3)
        # ======================================================================
        for bin_w in [0.10, 0.15]:
            prompt_f2 = (
                f"Write a self-contained Python script using ASE, NumPy, and Matplotlib to compute the planar-averaged "
                f"water mass density profile rho(z) in g/cm^3 (bin width {bin_w} Å) for O_w and H_w across {fp} "
                f"(fallback: {fb}). Separate {name} lattice oxygens from water oxygens via O-H covalent bonds (<= {oh_c} Å), "
                f"convert histogram counts to g/cm^3 using water molar mass 18.01528 g/mol and Avogadro's number 6.023e23, "
                f"and save the density profile plot to water_density_profile.png."
            )
            thinking_f2 = (
                f"1. Load trajectory frames from {fp} (fallback {fb}) and wrap coordinates under PBC.\n"
                f"2. Separate water oxygens O_w (bonded to H within {oh_c} Å) from {name} crystal oxygens.\n"
                f"3. Histogram z-coordinates with bin_width={bin_w} Å, convert counts to mass density (g/cm^3) via ((count / 6.023e23) * 18.01528) / (bin_volume * 1e-24), and save `water_density_profile.png`."
            )
            code_f2 = rf'''import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

file_path = "{fp}"
fallback_path = "{fb}"
if not os.path.exists(file_path):
    file_path = fallback_path

traj = read(file_path, index=':')
frames = traj if isinstance(traj, list) else [traj]
atoms0 = frames[0].copy()
atoms0.set_pbc(True)
symbols0 = np.array(atoms0.get_chemical_symbols())

# Separate water O_w from lattice O_slab
i_oh, _ = neighbor_list('ij', atoms0, cutoff={{('O', 'H'): {oh_c}}}, self_interaction=False)
o_w_indices = sorted(set(np.unique(i_oh)))
h_w_indices = sorted(np.where(symbols0 == 'H')[0].tolist())

cell = atoms0.get_cell()
bin_width = {bin_w}
area_xy = float(np.linalg.norm(np.cross(cell[0], cell[1])))
volume_bin = area_xy * bin_width
z_max = float(cell[2, 2])
bins = np.arange(0.0, z_max + bin_width, bin_width)
z_centers = 0.5 * (bins[:-1] + bins[1:])

ow_profiles, hw_profiles = [], []
for frame in frames:
    frame.wrap()
    pos = frame.get_positions()
    freq_ow, _ = np.histogram(pos[o_w_indices, 2], bins=bins)
    freq_hw, _ = np.histogram(pos[h_w_indices, 2], bins=bins)
    rho_ow = ((freq_ow / 6.023e23) * 18.01528) / (volume_bin * 1e-24)
    rho_hw = ((freq_hw / (2.0 * 6.023e23)) * 18.01528) / (volume_bin * 1e-24)
    ow_profiles.append(rho_ow)
    hw_profiles.append(rho_hw)

ow_mean = np.mean(ow_profiles, axis=0)
hw_mean = np.mean(hw_profiles, axis=0)

print(f"Frames analyzed: {{len(frames)}} | O_w count: {{len(o_w_indices)}} | H_w count: {{len(h_w_indices)}}")
print(f"Peak O_w Mass Density: {{ow_mean.max():.3f}} g/cm^3 at Z = {{z_centers[np.argmax(ow_mean)]:.2f}} Å")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(z_centers, ow_mean, lw=2.0, color='#1f77b4', label='Water O_w Density')
ax.plot(z_centers, hw_mean, lw=1.8, color='#2ca02c', linestyle='--', label='Water H_w Density')
ax.axhline(1.0, color='crimson', linestyle=':', lw=1.2, label='Bulk Water (1.0 g/cm^3)')
ax.set_xlabel('Z Coordinate (Å)')
ax.set_ylabel('Mass Density (g/cm^3)')
ax.set_title('Planar-Averaged Water Mass Density Profile: {name}')
ax.legend(loc='best')
ax.grid(True, linestyle='--', alpha=0.5)
plt.savefig('water_density_profile.png', dpi=300, bbox_inches='tight')
plt.close()'''

            sft_rows.append({
                "family": "F2_Water_Mass_Density_Profile",
                "difficulty_tier": 2,
                "prompt": prompt_f2,
                "thinking": thinking_f2,
                "code": code_f2,
            })

        # ======================================================================
        # FAMILY 3: Interfacial Pair Radial Distribution Functions g(r) (Pure ASE/NumPy)
        # ======================================================================
        prompt_f3 = (
            f"Write a complete Python script using ASE and NumPy (without pwtools) to compute and plot "
            f"the three interfacial pair radial distribution functions g(r) for {name}/Water from {fp} "
            f"(fallback: {fb}): (1) {m}_cus - O_w, (2) O_br - H_w, and (3) O_br - O_w up to r_max = 6.5 Å "
            f"with dr = 0.05 Å. Save the plot to interfacial_pair_rdfs.png."
        )
        thinking_f3 = (
            f"1. Load structure/trajectory from {fp} (fallback {fb}) and classify surface metal {m}_sur (z >= max(m_z) - 1.5 Å), bulk vs water oxygen, {m}_cus (within {cus_c} Å of O_w), and bridge oxygens O_br.\n"
            f"2. Compute pairwise minimum-image distances via `d_mat = frame.get_all_distances(mic=True)` and normalize spherical shell histograms by `(N_a * N_b / V_cell) * 4 * pi * r^2 * dr`.\n"
            f"3. Plot all three g(r) curves with plain Unicode labels and save `interfacial_pair_rdfs.png`."
        )
        code_f3 = rf'''import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

file_path = "{fp}"
fallback_path = "{fb}"
if not os.path.exists(file_path):
    file_path = fallback_path

traj = read(file_path, index=':')
frames = traj if isinstance(traj, list) else [traj]
atoms0 = frames[0].copy()
atoms0.wrap()
atoms0.set_pbc(True)
pos0 = atoms0.get_positions()
sym0 = np.array(atoms0.get_chemical_symbols())
cell_vol = float(atoms0.get_volume())

# Separate water O_w from crystal lattice O_slab via covalent O-H bonds
i_oh, _ = neighbor_list('ij', atoms0, cutoff={{('O', 'H'): {oh_c}}}, self_interaction=False)
o_w_set = set(np.unique(i_oh))
o_w_index = sorted(o_w_set)
h_w_index = sorted(np.where(sym0 == 'H')[0].tolist())
o_bul_index = sorted([i for i, s in enumerate(sym0) if s == 'O' and i not in o_w_set], key=lambda idx: pos0[idx, 2])

m_indices = [i for i, s in enumerate(sym0) if s == '{m}']
m_z_max = float(pos0[m_indices, 2].max())
m_sur_index = [i for i in m_indices if pos0[i, 2] >= m_z_max - 1.5]

m_cus_index = []
for i_m in m_sur_index:
    for j_ow in o_w_index:
        if atoms0.get_distance(i_m, j_ow, mic=True) <= {cus_c}:
            if i_m not in m_cus_index:
                m_cus_index.append(i_m)
if not m_cus_index:
    m_cus_index = m_sur_index

n_br = max(1, len(m_cus_index))
o_br_index = o_bul_index[-n_br:]

dr = 0.05
rmax = 6.5
r_bins = np.arange(0.0, rmax + dr, dr)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
shell_vol = 4.0 * np.pi * (r_centers ** 2) * dr

def compute_pair_rdf_matrix(d_matrix, idx_a, idx_b):
    if not idx_a or not idx_b:
        return np.zeros_like(r_centers)
    sub_d = d_matrix[np.ix_(idx_a, idx_b)].ravel()
    sub_d = sub_d[sub_d > 1e-5]
    counts, _ = np.histogram(sub_d, bins=r_bins)
    norm = (len(idx_a) * len(idx_b) / cell_vol) * shell_vol
    return counts / np.maximum(norm, 1e-12)

g_mow_all, g_obrh_all, g_obrow_all = [], [], []
for frame in frames:
    frame.set_pbc(True)
    d_mat = frame.get_all_distances(mic=True)
    g_mow_all.append(compute_pair_rdf_matrix(d_mat, m_cus_index, o_w_index))
    g_obrh_all.append(compute_pair_rdf_matrix(d_mat, o_br_index, h_w_index))
    g_obrow_all.append(compute_pair_rdf_matrix(d_mat, o_br_index, o_w_index))

g_mow = np.mean(g_mow_all, axis=0)
g_obrh = np.mean(g_obrh_all, axis=0)
g_obrow = np.mean(g_obrow_all, axis=0)

print(f"CUS {m} sites: {{len(m_cus_index)}} | Bridge O_br sites: {{len(o_br_index)}} | Water O_w: {{len(o_w_index)}}")
print(f"First peak {m}_cus-O_w g(r) = {{g_mow.max():.2f}} at r = {{r_centers[np.argmax(g_mow)]:.2f}} Å")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(r_centers, g_mow, lw=2.0, color='#d35400', label='{m}_cus - O_w')
ax.plot(r_centers, g_obrh, lw=2.0, color='#2980b9', label='O_br - H_w')
ax.plot(r_centers, g_obrow, lw=2.0, color='#8e44ad', label='O_br - O_w')
ax.set_xlabel('Distance r (Å)')
ax.set_ylabel('Pair RDF g(r)')
ax.set_title('Interfacial Pair Radial Distribution Functions: {name}')
ax.set_xlim(0.0, rmax)
ax.legend(loc='best')
ax.grid(True, linestyle='--', alpha=0.5)
plt.savefig('interfacial_pair_rdfs.png', dpi=300, bbox_inches='tight')
plt.close()'''

        rejected_f3 = rf'''from pwtools.crys import rpdf, atoms2struct
from ase.io import read
# Crash: pwtools is not installed in the sandbox container!
atoms = read("{fp}", index=0)
res = rpdf(atoms2struct(atoms), dr=0.05, amask=['{m}', 'O'])'''

        sft_rows.append({
            "family": "F3_Pair_RDF_Analysis",
            "difficulty_tier": 2,
            "prompt": prompt_f3,
            "thinking": thinking_f3,
            "code": code_f3,
        })
        dpo_rows.append({
            "family": "F3_Pair_RDF_Analysis",
            "error_type": "pwtools_uninstalled_import",
            "prompt": prompt_f3,
            "chosen": f"<thinking>\n{thinking_f3}\n</thinking>\n\n```python\n{code_f3}\n```",
            "rejected": f"```python\n{rejected_f3}\n```",
        })

        # ======================================================================
        # FAMILY 4: Surface Hydroxylation Fractions & Continuous Survival Probability P(t)
        # ======================================================================
        prompt_f4 = (
            f"Write a Python script using ASE and NumPy to track adsorbed water speciation (dissociated OH vs intact H2O "
            f"using O-H cutoff 1.2 Å) and continuous survival probability P(t) via multi-frame set intersections on {name} "
            f"({fp}, fallback: {fb}). Integrate P(t) using trapezoidal integration to report residence times tau_OH and tau_H2O, "
            f"and save survival_probability.png."
        )
        thinking_f4 = (
            f"1. Load trajectory frames from {fp} (fallback {fb}) and identify adsorbed water oxygens O_adb within {cus_c} Å of surface {m} sites.\n"
            f"2. Per frame, count H neighbors within 1.2 Å via `i_idx, _, _ = neighbor_list('ijd', frame, cutoff={{('O', 'H'): 1.2}}, self_interaction=False)` and `np.bincount`.\n"
            f"3. Compute continuous survival probability P(tau) via `set.intersection(*sets[j:j+tau+1])` and integrate residence time via `getattr(np, 'trapezoid', np.trapz)`."
        )
        code_f4 = rf'''import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

file_path = "{fp}"
fallback_path = "{fb}"
if not os.path.exists(file_path):
    file_path = fallback_path

raw_traj = read(file_path, index=':')
frames = raw_traj if isinstance(raw_traj, list) else [raw_traj]
atoms0 = frames[0].copy()
atoms0.wrap()
atoms0.set_pbc(True)
pos0 = atoms0.get_positions()
sym0 = np.array(atoms0.get_chemical_symbols())

i_oh0, _ = neighbor_list('ij', atoms0, cutoff={{('O', 'H'): {oh_c}}}, self_interaction=False)
o_w_set = set(np.unique(i_oh0))
m_indices = [i for i, s in enumerate(sym0) if s == '{m}']
m_sur_idx = [i for i in m_indices if pos0[i, 2] >= pos0[m_indices, 2].max() - 1.5]

o_adb_idx = sorted({{
    j_ow for i_m in m_sur_idx for j_ow in o_w_set
    if atoms0.get_distance(i_m, j_ow, mic=True) <= 3.2
}})
if not o_adb_idx:
    o_adb_idx = sorted(o_w_set)

indx_oh, indx_h2o, frac_oh, frac_h2o = [], [], [], []
num_ads = max(1, len(o_adb_idx))

for frame in frames:
    frame.wrap()
    frame.set_pbc(True)
    i_idx, _, _ = neighbor_list('ijd', frame, cutoff={{('O', 'H'): 1.25}}, self_interaction=False)
    oh_counts = np.bincount(i_idx, minlength=len(frame))
    oh_f = [idx for idx in o_adb_idx if oh_counts[idx] == 1]
    h2o_f = [idx for idx in o_adb_idx if oh_counts[idx] == 2]
    indx_oh.append(oh_f)
    indx_h2o.append(h2o_f)
    frac_oh.append(100.0 * len(oh_f) / num_ads)
    frac_h2o.append(100.0 * len(h2o_f) / num_ads)

oh_sets = [set(f) for f in indx_oh]
h2o_sets = [set(f) for f in indx_h2o]
num_f = len(frames)
tau_range = np.linspace(0, num_f - 1, min(101, max(2, num_f)), dtype=int)
dt_ns = 0.0005
times_ns = tau_range * dt_ns

surv_oh, surv_h2o = [], []
for tau in tau_range:
    if tau == 0:
        surv_oh.append(1.0)
        surv_h2o.append(1.0)
    else:
        s_oh = [len(set.intersection(*oh_sets[j:j+tau+1])) / len(oh_sets[j]) for j in range(num_f - tau) if oh_sets[j]]
        s_h2o = [len(set.intersection(*h2o_sets[j:j+tau+1])) / len(h2o_sets[j]) for j in range(num_f - tau) if h2o_sets[j]]
        surv_oh.append(float(np.mean(s_oh)) if s_oh else 0.0)
        surv_h2o.append(float(np.mean(s_h2o)) if s_h2o else 0.0)

trapz_fn = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
def calc_tau(surv_arr, t_arr):
    if len(t_arr) < 2:
        return 0.0
    denom = float(trapz_fn(surv_arr, t_arr))
    return float(trapz_fn(np.array(surv_arr) * t_arr, t_arr) / denom) if denom > 1e-12 else 0.0

tau_oh = calc_tau(surv_oh, times_ns)
tau_h2o = calc_tau(surv_h2o, times_ns)
print(f"Adsorbed O_adb sites tracked: {{len(o_adb_idx)}}")
print(f"Mean Dissociated OH Fraction: {{np.mean(frac_oh):.2f}}% | Mean Intact H2O Fraction: {{np.mean(frac_h2o):.2f}}%")
print(f"Residence Time tau_OH: {{tau_oh:.4f}} ns | Residence Time tau_H2O: {{tau_h2o:.4f}} ns")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(times_ns, surv_oh, lw=2.0, color='#2c3e50', label='Dissociated OH')
ax.plot(times_ns, surv_h2o, lw=2.0, color='#c0392b', label='Molecular H2O')
ax.set_xlabel('Time Lag (ns)')
ax.set_ylabel('Survival Probability P(t)')
ax.set_title('Interfacial Water Survival Probability: {name}')
ax.legend(loc='best')
ax.grid(True, linestyle='--', alpha=0.5)
plt.savefig('survival_probability.png', dpi=300, bbox_inches='tight')
plt.close()'''

        sft_rows.append({
            "family": "F4_Survival_Probability_Kinetics",
            "difficulty_tier": 2,
            "prompt": prompt_f4,
            "thinking": thinking_f4,
            "code": code_f4,
        })

        # ======================================================================
        # FAMILY 5: Full 6-Stage Multi-Model DP Ensemble Interfacial Pipeline
        # ======================================================================
        prompt_f5 = (
            f"Write a complete, self-contained scientific Python script using ASE, NumPy, and Matplotlib (Agg backend, without pwtools) "
            f"to analyze an interfacial {name}/H2O trajectory (metal '{m}', stoichiometry s={s}) across dp=3 DeepMD models "
            f"(fallback: {fp} or {fb}). Classify M_sur, O_bulk, O_w, M_cus, O_adb, O_br; compute z-mass density (g/cm^3), "
            f"3 pair RDFs ({m}_cus-O_w, O_br-H_w, O_br-O_w), surface hydroxylation fractions (OH vs H2O, cutoff 1.2 Å), "
            f"and continuous survival probability P(t); save simulation_results.npz, survival_fraction_results.npz, "
            f"full_analysis_presentation.png (2x2 figure with +/- 1 std shading), and summary_report.txt."
        )
        thinking_f5 = (
            f"1. Loop over dp=3 models with fallback to `{fp}` / `{fb}` and unpack trajectory frames.\n"
            f"2. Classify {m}_sur (z >= z_max - 1.5 Å), O_bulk (lowest s*N_{m} oxygens), O_w, {m}_cus & O_adb (<= {cus_c} Å), and O_br.\n"
            f"3. Compute mass density (g/cm^3), spherical-shell normalized pair RDFs via `get_all_distances(mic=True)`, OH/H2O fractions, and P(t) via set intersections, then save both .npz files, the 2x2 plot, and summary_report.txt."
        )
        code_f5 = rf"""import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io import read
from ase.neighborlist import neighbor_list

dp = 3
s = {s}
m = '{m}'
base_dir = globals().get('base_dir', '.')
analysis_stride = 10
bin_width = 0.10
dr = 0.05
rmax = 6.5
r_bins = np.arange(0.0, rmax + dr, dr)
r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
shell_vol = 4.0 * np.pi * (r_centers ** 2) * dr

histo_y_ow_dpmd, histo_y_hw_dpmd, histbin_dpmd_all = [], [], []
g_r_mow_dpmd, g_r_osh_dpmd, g_r_osow_dpmd, rd_dpmd_all = [], [], [], []
all_surv_oh, all_surv_h2o, all_times_survival = [], [], []
all_diss_frac, all_intact_frac = [], []
detected_dt_ns = 0.0005

for mod_idx in range(1, dp + 1):
    cand = os.path.join(base_dir, str(mod_idx), 'iro2_water_concat.extxyz')
    frames = None
    for p in [cand, "{fp}", "{fb}"]:
        if os.path.exists(p):
            raw = read(p, index=slice(0, None, analysis_stride))
            frames = raw if isinstance(raw, list) else [raw]
            break
    if not frames:
        continue

    atoms0 = frames[0].copy()
    atoms0.wrap()
    atoms0.set_pbc(True)
    pos0 = atoms0.get_positions()
    sym0 = np.array(atoms0.get_chemical_symbols())
    cell = atoms0.get_cell()
    vol_bin = float(cell[0, 0] * cell[1, 1] * bin_width)
    cell_vol = float(atoms0.get_volume())

    i_oh0, _ = neighbor_list('ij', atoms0, cutoff={{('O', 'H'): {oh_c}}}, self_interaction=False)
    o_w_set = set(np.unique(i_oh0))
    o_w_index = sorted(o_w_set)
    h_w_index = sorted(np.where(sym0 == 'H')[0].tolist())
    o_bul_index = sorted([i for i, el in enumerate(sym0) if el == 'O' and i not in o_w_set], key=lambda idx: pos0[idx, 2])

    m_indices = [i for i, el in enumerate(sym0) if el not in ('O', 'H')]
    m_sur_index = [i for i in m_indices if pos0[i, 2] >= pos0[m_indices, 2].max() - 1.5]
    m_cus_index = [i_m for i_m in m_sur_index if any(atoms0.get_distance(i_m, j_ow, mic=True) <= {cus_c} for j_ow in o_w_index)] or m_sur_index
    o_adb_index = sorted({{j_ow for i_m in m_sur_index for j_ow in o_w_index if atoms0.get_distance(i_m, j_ow, mic=True) <= 3.2}}) or o_w_index
    o_sur_index = o_bul_index[-max(1, len(m_cus_index)):]

    bins = np.linspace(float(pos0[:, 2].min()), float(cell[2, 2]) + 0.5, max(10, int((float(cell[2, 2]) + 0.5 - float(pos0[:, 2].min())) / bin_width)))
    histo_ow, histo_hw, g_mow, g_osh, g_osow = [], [], [], [], []
    indx_oh, indx_h2o, frac_oh, frac_h2o = [], [], [], []
    num_ads = max(1, len(o_adb_index))

    for frame in frames:
        frame.wrap()
        frame.set_pbc(True)
        f_pos = frame.get_positions()
        freq_ow, _ = np.histogram(f_pos[o_w_index, 2], bins=bins)
        freq_hw, _ = np.histogram(f_pos[h_w_index, 2], bins=bins)
        histo_ow.append(((freq_ow / 6.023e23) * 18.01528) / (vol_bin * 1e-24))
        histo_hw.append(((freq_hw / (2.0 * 6.023e23)) * 18.01528) / (vol_bin * 1e-24))

        d_mat = frame.get_all_distances(mic=True)
        for idx_a, idx_b, target_g in [(m_cus_index, o_w_index, g_mow), (o_sur_index, h_w_index, g_osh), (o_sur_index, o_w_index, g_osow)]:
            sub_d = d_mat[np.ix_(idx_a, idx_b)].ravel()
            sub_d = sub_d[sub_d > 1e-5]
            c_hist, _ = np.histogram(sub_d, bins=r_bins)
            norm = (len(idx_a) * len(idx_b) / cell_vol) * shell_vol
            target_g.append(c_hist / np.maximum(norm, 1e-12))

        i_idx, _, _ = neighbor_list('ijd', frame, cutoff={{('O', 'H'): 1.25}}, self_interaction=False)
        oh_c_arr = np.bincount(i_idx, minlength=len(frame))
        oh_f = [idx for idx in o_adb_index if oh_c_arr[idx] == 1]
        h2o_f = [idx for idx in o_adb_index if oh_c_arr[idx] == 2]
        indx_oh.append(oh_f); indx_h2o.append(h2o_f)
        frac_oh.append(100.0 * len(oh_f) / num_ads)
        frac_h2o.append(100.0 * len(h2o_f) / num_ads)

    oh_sets, h2o_sets = [set(x) for x in indx_oh], [set(x) for x in indx_h2o]
    num_f = len(frames)
    tau_range = np.linspace(0, num_f - 1, min(101, max(2, num_f)), dtype=int)
    s_oh, s_h2o = [], []
    for tau in tau_range:
        if tau == 0:
            s_oh.append(1.0); s_h2o.append(1.0)
        else:
            so = [len(set.intersection(*oh_sets[j:j+tau+1])) / len(oh_sets[j]) for j in range(num_f - tau) if oh_sets[j]]
            sh = [len(set.intersection(*h2o_sets[j:j+tau+1])) / len(h2o_sets[j]) for j in range(num_f - tau) if h2o_sets[j]]
            s_oh.append(float(np.mean(so)) if so else 0.0)
            s_h2o.append(float(np.mean(sh)) if sh else 0.0)

    cent = 0.5 * (bins[0] + bins[-1])
    histo_y_ow_dpmd.append(np.mean(histo_ow, axis=0))
    histo_y_hw_dpmd.append(np.mean(histo_hw, axis=0))
    histbin_dpmd_all.append(bins[:-1] - cent)
    g_r_mow_dpmd.append(np.mean(g_mow, axis=0))
    g_r_osh_dpmd.append(np.mean(g_osh, axis=0))
    g_r_osow_dpmd.append(np.mean(g_osow, axis=0))
    rd_dpmd_all.append(r_centers)
    all_surv_oh.append(s_oh); all_surv_h2o.append(s_h2o)
    all_times_survival.append(tau_range * detected_dt_ns)
    all_diss_frac.append(frac_oh); all_intact_frac.append(frac_h2o)

np.savez('simulation_results.npz', m_symbol=m, ow_y=histo_y_ow_dpmd, hw_y=histo_y_hw_dpmd,
         bins_z=histbin_dpmd_all, rd=rd_dpmd_all, g_mow=g_r_mow_dpmd, g_osh=g_r_osh_dpmd, g_osow=g_r_osow_dpmd)
np.savez('survival_fraction_results.npz', m_symbol=m, times_surv=all_times_survival,
         surv_oh=all_surv_oh, surv_h2o=all_surv_h2o, frac_oh=all_diss_frac, frac_h2o=all_intact_frac, dt_ns=detected_dt_ns)

fig, axs = plt.subplots(2, 2, figsize=(14, 11), dpi=300)
axs[0, 0].plot(histbin_dpmd_all[0], np.mean(histo_y_ow_dpmd, axis=0), label='O_w')
axs[0, 0].set_title('A) Water Mass Density Profile')
axs[0, 1].plot(r_centers, np.mean(g_r_mow_dpmd, axis=0), label=f'{m}_cus-O_w')
axs[0, 1].set_title('B) Pair RDFs')
axs[1, 0].plot(all_times_survival[0], np.mean(all_surv_oh, axis=0), label='OH')
axs[1, 0].set_title('C) Survival Probability P(t)')
axs[1, 1].plot(np.mean(all_diss_frac, axis=0), label='Dissociated %')
axs[1, 1].set_title('D) Hydroxylation Fraction')
plt.tight_layout()
plt.savefig('full_analysis_presentation.png', dpi=300, bbox_inches='tight')
plt.close()

with open('summary_report.txt', 'w') as f:
    f.write(f"=== FINAL ANALYSIS REPORT: {{m}} ===\\nAvg Dissociated: {{np.mean(all_diss_frac):.2f}}%\\n")
print("Saved simulation_results.npz, survival_fraction_results.npz, full_analysis_presentation.png, and summary_report.txt")"""

        sft_rows.append({
            "family": "F5_Full_6Stage_Ensemble_Analysis",
            "difficulty_tier": 3,
            "prompt": prompt_f5,
            "thinking": thinking_f5,
            "code": code_f5,
        })

    # ==========================================================================
    # FAMILY 6: ASE Molecular Dynamics Workflows (Langevin NVT, VelocityVerlet, MaxwellBoltzmann)
    # ==========================================================================
    for temp_k, steps_n, gamma_val in [(300.0, 50, 0.01), (330.0, 100, 0.02), (350.0, 80, 0.005)]:
        prompt_f6 = (
            f"Write an executable Python script using ASE (`ase.md`) that loads frame 0 of "
            f"/mnt/data/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz (fallback: public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz), "
            f"freezes the bottom TiO2 crystal layer (z <= z_cryst_min + 1.5 Å) using `FixAtoms`, initializes velocities "
            f"at {temp_k} K via `MaxwellBoltzmannDistribution` (with `Stationary` and `ZeroRotation`), sets up a `Langevin` "
            f"integrator (timestep 1.0 fs, friction {gamma_val} fs^-1), and logs the instantaneous temperature."
        )
        thinking_f6 = (
            f"1. Load structure from fallback path, enable PBC, and identify bottom crystal slab atoms (`z <= z_cryst_min + 1.5` Å) to freeze with `FixAtoms`.\n"
            f"2. Initialize velocities at {temp_k} K using `MaxwellBoltzmannDistribution(atoms, temperature_K={temp_k})`, `Stationary(atoms)`, and `ZeroRotation(atoms)`.\n"
            f"3. Instantiate `Langevin(atoms, timestep=1.0 * units.fs, temperature_K={temp_k}, friction={gamma_val} / units.fs)` and print kinetic temperature."
        )
        code_f6 = rf'''import os
import numpy as np
from ase import units
from ase.io import read
from ase.constraints import FixAtoms
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase.md.langevin import Langevin

file_path = "/mnt/data/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz"
fallback_path = "public/Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz"
if not os.path.exists(file_path):
    file_path = fallback_path

atoms = read(file_path, index=0)
atoms.set_pbc(True)
positions = atoms.get_positions()
symbols = np.array(atoms.get_chemical_symbols())

# Freeze bottom TiO2 layer based on Ti z-coordinates
ti_z = positions[symbols == 'Ti', 2]
z_freeze_cut = float(ti_z.min()) + 1.5
freeze_mask = (symbols != 'H') & (positions[:, 2] <= z_freeze_cut)
atoms.set_constraint(FixAtoms(mask=freeze_mask))

# Initialize Maxwell-Boltzmann velocities at {temp_k} K
MaxwellBoltzmannDistribution(atoms, temperature_K={temp_k})
Stationary(atoms)
ZeroRotation(atoms)

dyn = Langevin(
    atoms,
    timestep=1.0 * units.fs,
    temperature_K={temp_k},
    friction={gamma_val} / units.fs,
)
print(f"Frozen bottom slab atoms: {{int(np.sum(freeze_mask))}} / {{len(atoms)}}")
print(f"Initialized kinetic temperature: {{atoms.get_temperature():.2f}} K (Target: {temp_k} K)")'''

        sft_rows.append({
            "family": "F6_ASE_Molecular_Dynamics",
            "difficulty_tier": 1,
            "prompt": prompt_f6,
            "thinking": thinking_f6,
            "code": code_f6,
        })

    # ==========================================================================
    # FAMILY 7: ASE Surface Slab Building, Adsorbates & Geometry Optimization
    # ==========================================================================
    for metal_el, a_lat, vac_a in [("Pt", 3.943, 12.0), ("Cu", 3.615, 14.0), ("Au", 4.078, 15.0)]:
        prompt_f7 = (
            f"Write a Python script using `ase.build` and `ase.constraints` to construct a 4-layer {metal_el}(111) "
            f"surface slab (3x3 supercell, lattice constant a={a_lat} Å, vacuum={vac_a} Å), add an H2O adsorbate "
            f"2.2 Å above the ontop site, freeze the bottom 2 metal layers with `FixAtoms`, and save `{metal_el.lower()}111_h2o.extxyz`."
        )
        thinking_f7 = (
            f"1. Build {metal_el}(111) slab with `fcc111('{metal_el}', size=(3, 3, 4), a={a_lat}, vacuum={vac_a})` and set PBC.\n"
            f"2. Add water molecule via `add_adsorbate(slab, molecule('H2O'), height=2.2, position='ontop')`.\n"
            f"3. Freeze bottom 2 metal layers using `slab.get_tags() >= 3` (in ASE `fcc111`, layer tags are 1=top to 4=bottom) or z-coordinate threshold, and write `{metal_el.lower()}111_h2o.extxyz`."
        )
        code_f7 = rf'''import numpy as np
from ase.build import fcc111, molecule, add_adsorbate
from ase.constraints import FixAtoms
from ase.io import write

slab = fcc111('{metal_el}', size=(3, 3, 4), a={a_lat}, vacuum={vac_a})
slab.set_pbc([True, True, True])

h2o = molecule('H2O')
add_adsorbate(slab, h2o, height=2.2, position='ontop')

positions = slab.get_positions()
symbols = np.array(slab.get_chemical_symbols())
metal_z = sorted(np.unique(np.round(positions[symbols == '{metal_el}', 2], 2)))
z_cut = metal_z[1] + 0.3
freeze_mask = (symbols == '{metal_el}') & (positions[:, 2] <= z_cut)
slab.set_constraint(FixAtoms(mask=freeze_mask))

write('{metal_el.lower()}111_h2o.extxyz', slab, format='extxyz')
print(f"Built {metal_el}(111)+H2O slab: {{len(slab)}} atoms | Frozen bottom 2 layers: {{int(np.sum(freeze_mask))}} atoms")'''

        sft_rows.append({
            "family": "F7_ASE_Slab_Builder_Constraints",
            "difficulty_tier": 1,
            "prompt": prompt_f7,
            "thinking": thinking_f7,
            "code": code_f7,
        })

    # ==========================================================================
    # FAMILY 8: Production LAMMPS + DeePMD Restart-Linked Batch Chaining (`Molecular Dynamics/`)
    # ==========================================================================
    for total_ps, batch_ps, n_graphs in [(5000, 500, 3), (2000, 250, 4)]:
        prompt_f8 = (
            f"Write a Python script that generates restart-linked LAMMPS production MD batch directories "
            f"(`batch_1` to `batch_{total_ps // batch_ps}`, total {total_ps} ps, chunk {batch_ps} ps, dt=0.0005 ps) "
            f"and a SLURM dependency-chaining `submit_from.sh` script (`sbatch --parsable --dependency=afterok:$PREV_JOBID`) "
            f"for liquid Water at 300 K (`simple` style: plugin load libdeepmd_lmp.so, masses 1=2.014, 2=15.999, "
            f"pair_style deepmd with {n_graphs} frozen graphs and out_file md.out out_freq 100) and Graphene-Water Interface at 330 K (`complex` style)."
        )
        thinking_f8 = (
            f"1. Compute `num_batches = {total_ps} // {batch_ps}` and `batch_steps = round({batch_ps} / 0.0005)`.\n"
            f"2. For batch 1, emit `read_data` and `velocity all create`; for batch >= 2, emit `read_restart ../batch_{{i-1}}/lammps.restart`.\n"
            f"3. Write `submit_from.sh` chaining `sbatch --parsable --dependency=afterok:$PREV_JOBID` across all batches."
        )
        code_f8 = rf'''import os
from pathlib import Path

total_time_ps = {total_ps}
batch_time_ps = {batch_ps}
dt = 0.0005
graph_num = {n_graphs}
num_batches = int(total_time_ps // batch_time_ps)
batch_steps = int(round(batch_time_ps / dt))

out_root = Path("Long_Run_Generated/DPMD_Water")
out_root.mkdir(parents=True, exist_ok=True)
graphs = [f"../../Frozen_Graphs/DPMD_Water/{{i}}/graph{{i}}.pb" for i in range(1, graph_num + 1)]

for b_idx in range(1, num_batches + 1):
    b_dir = out_root / f"batch_{{b_idx}}"
    b_dir.mkdir(exist_ok=True)
    read_cmd = (
        "read_data       ../../Starting_Configurations/Water_DPMD_Validation.lmp"
        if b_idx == 1 else
        f"read_restart    ../batch_{{b_idx - 1}}/lammps.restart"
    )
    vel_cmd = "velocity        all create 300 12345678\n" if b_idx == 1 else ""
    in_lammps = (
        "plugin load     libdeepmd_lmp.so\\n"
        "units           metal\\n"
        "boundary        p p p\\n"
        "atom_style      atomic\\n"
        "neighbor        2.0 bin\\n"
        "neigh_modify    every 10 delay 0 check no\\n"
        f"{{read_cmd}}\\n"
        "mass            1 2.014\\n"
        "mass            2 15.999\\n"
        f"pair_style      deepmd {{' '.join(graphs)}} out_file md.out out_freq 100\\n"
        "pair_coeff      * *\\n"
        f"{{vel_cmd}}"
        "fix             1 all nvt temp 300 300 0.1\\n"
        f"timestep        {{dt}}\\n"
        "thermo_style    custom step pe ke etotal temp press vol\\n"
        "thermo          100\\n"
        f"dump            1 all dcd 100 DPMD_Water_batch_{{b_idx}}.dcd\\n"
        "timer timeout   23:55:00 every 100\\n"
        f"run             {{batch_steps}}\\n"
        "write_restart   lammps.restart\\n"
    )
    (b_dir / "in.lammps").write_text(in_lammps)

submit_sh = (
    "#!/bin/bash\\nset -e\\n"
    "COMBO_DIR=\"$(cd \"$(dirname \"$0\")\" && pwd)\"\n"
    f"NUM_BATCHES={{num_batches}}\\n"
    "START=${{1:-1}}\nPREV_JOBID=\"\"\n"
    "for ((i=START; i<=NUM_BATCHES; i++)); do\\n"
    "  BATCH_DIR=\"$COMBO_DIR/batch_$i\"\n"
    "  if [ -z \"$PREV_JOBID\" ]; then\\n"
    "    JOBID=$(sbatch --parsable --chdir=\"$BATCH_DIR\" \"$BATCH_DIR/lmp.sub\")\n"
    "  else\\n"
    "    JOBID=$(sbatch --parsable --dependency=afterok:$PREV_JOBID --chdir=\"$BATCH_DIR\" \"$BATCH_DIR/lmp.sub\")\n"
    "  fi\\n"
    "  PREV_JOBID=$JOBID\\n"
    "done\\n"
)
(out_root / "submit_from.sh").write_text(submit_sh)
print(f"Generated {{num_batches}} LAMMPS batches ({{batch_steps}} steps/batch) and submit_from.sh")'''

        sft_rows.append({
            "family": "F8_LAMMPS_DeePMD_Batch_Chaining",
            "difficulty_tier": 2,
            "prompt": prompt_f8,
            "thinking": thinking_f8,
            "code": code_f8,
        })

    # ==========================================================================
    # FAMILY 9: DeePMD-kit Raw Data Conversion & se_e2_a input.json (`Training/`)
    # ==========================================================================
    for type_map_cfg, sel_cfg in [(["H", "O"], [120, 64]), (["Ti", "O", "H"], [60, 128, 160]), (["Ir", "O", "H"], [46, 92, 92])]:
        prompt_f9 = (
            f"Write a Python script that creates a leak-free train/Test split from DeePMD `.raw` files "
            f"(`coord.raw`, `box.raw`, `energy.raw`, `force.raw`, `type.raw`), converts chunks into `set.000/*.npy` "
            f"float32 binary arrays, and writes a valid DeePMD-kit `se_e2_a` `input.json` configuration for "
            f"`type_map = {type_map_cfg}` and `sel = {sel_cfg}`."
        )
        thinking_f9 = (
            f"1. Verify `.raw` shapes (`box.raw`: N x 9, `coord.raw`: N x 3*N_atoms, `energy.raw`: N x 1, `force.raw`: N x 3*N_atoms) and partition indices into disjoint train and test sets.\n"
            f"2. Save float32 `.npy` arrays inside `set.000/` and output `input.json` with `se_e2_a` (`rcut_smth=3.0`, `rcut=6.0`, `neuron=[120, 120, 120]`, `axis_neuron=12`)."
        )
        code_f9 = rf'''import json
import numpy as np
from pathlib import Path

work_dir = Path("deepmd_dataset_demo")
(work_dir / "set.000").mkdir(parents=True, exist_ok=True)
(work_dir / "Test" / "set.000").mkdir(parents=True, exist_ok=True)

# Create synthetic 20-frame raw arrays (10 atoms) and perform disjoint train/test split
n_frames, n_atoms = 20, 10
rng = np.random.default_rng(42)
 raw_arrays = {{
    "box": rng.uniform(10.0, 12.0, size=(n_frames, 9)).astype(np.float32),
    "coord": rng.uniform(0.0, 10.0, size=(n_frames, 3 * n_atoms)).astype(np.float32),
    "energy": rng.normal(-500.0, 1.5, size=(n_frames, 1)).astype(np.float32),
    "force": rng.normal(0.0, 0.2, size=(n_frames, 3 * n_atoms)).astype(np.float32),
}}
all_idx = np.arange(n_frames)
test_mask = np.zeros(n_frames, dtype=bool)
test_mask[:4] = True
train_mask = ~test_mask

for name, arr in raw_arrays.items():
    np.save(work_dir / "set.000" / f"{{name}}.npy", arr[train_mask])
    np.save(work_dir / "Test" / "set.000" / f"{{name}}.npy", arr[test_mask])

input_config = {{
    "model": {{
        "type_map": {type_map_cfg},
        "descriptor": {{
            "type": "se_e2_a",
            "sel": {sel_cfg},
            "rcut_smth": 3.0,
            "rcut": 6.0,
            "neuron": [120, 120, 120],
            "resnet_dt": False,
            "axis_neuron": 12,
            "seed": 1
        }},
        "fitting_net": {{"neuron": [120, 120, 120], "resnet_dt": True, "seed": 1}}
    }},
    "learning_rate": {{"type": "exp", "start_lr": 0.005, "decay_steps": 5000, "decay_rate": 0.95}},
    "loss": {{"type": "ener", "start_pref_e": 0.02, "limit_pref_e": 8, "start_pref_f": 1000, "limit_pref_f": 1}},
    "training": {{
        "stop_batch": 1000000,
        "batch_size": 1,
        "disp_file": "lcurve.out",
        "disp_freq": 100,
        "save_freq": 1000,
        "training_data": {{"systems": ["."], "set_prefix": "set", "batch_size": "auto"}},
        "validation_data": {{"systems": ["./Test"], "set_prefix": "set", "batch_size": "auto"}}
    }}
}}
(work_dir / "input.json").write_text(json.dumps(input_config, indent=2))
print(f"Saved leak-free train ({{int(train_mask.sum())}} frames) and test ({{int(test_mask.sum())}} frames) sets + input.json")'''
        # Remove accidental leading space on raw_arrays
        code_f9 = code_f9.replace(" raw_arrays =", "raw_arrays =")
        sft_rows.append({
            "family": "F9_DeePMD_Dataset_And_Config",
            "difficulty_tier": 2,
            "prompt": prompt_f9,
            "thinking": thinking_f9,
            "code": code_f9,
        })

    # ==========================================================================
    # FAMILY 10: Quantum ESPRESSO (`pw.x`) DFT Convergence Sweeps (`DFT/`)
    # ==========================================================================
    prompt_f10 = (
        "Write a Python script using ASE (`ase.build.bulk`, `ase.io.write` with `format='espresso-in'`) "
        "that generates Quantum ESPRESSO SCF input files (`pw.in`) for fcc Platinum (a=3.943 Å, Pt_ONCV_PBE-1.2.upf) "
        "across ecutwfc = [60, 80, 100] Ry with ecutrho = 4 * ecutwfc, Marzari-Vanderbilt cold smearing (degauss=0.02 Ry), "
        "and k-point mesh (4, 4, 4)."
    )
    thinking_f10 = (
        "1. Build fcc Pt cell via `bulk('Pt', 'fcc', a=3.943, cubic=True)` (4 atoms).\n"
        "2. Loop over `ecut in [60, 80, 100]`, set `ecutrho = 4 * ecut` (norm-conserving ONCV rule), and write `pw.in` via `ase.io.write(..., format='espresso-in')`."
    )
    code_f10 = r'''import os
from pathlib import Path
from ase.build import bulk
from ase.io import write

atoms = bulk('Pt', 'fcc', a=3.943, cubic=True)
pseudopotentials = {'Pt': 'Pt_ONCV_PBE-1.2.upf'}
base_out = Path('qe_pt_cutoff_sweep')

for ecut in [60, 80, 100]:
    job_dir = base_out / str(ecut)
    job_dir.mkdir(parents=True, exist_ok=True)
    input_qe = {
        'control': {
            'calculation': 'scf',
            'outdir': './outdir',
            'pseudo_dir': './../../pseudo/',
            'tprnfor': True,
            'disk_io': 'none',
        },
        'system': {
            'ibrav': 0,
            'nat': len(atoms),
            'ntyp': 1,
            'ecutwfc': ecut,
            'ecutrho': 4 * ecut,
            'input_dft': 'PBE',
            'nbnd': 44,
            'occupations': 'smearing',
            'smearing': 'marzari-vanderbilt',
            'degauss': 0.02,
        },
        'electrons': {
            'conv_thr': 1.0e-8,
            'mixing_beta': 0.1,
        },
    }
    write(job_dir / 'pw.in', atoms, format='espresso-in', input_data=input_qe, pseudopotentials=pseudopotentials, kpts=(4, 4, 4))
print("Generated QE pw.in files for ecutwfc = [60, 80, 100] Ry (ecutrho = 4 * ecutwfc)")'''

    sft_rows.append({
        "family": "F10_Quantum_ESPRESSO_DFT_Sweep",
        "difficulty_tier": 1,
        "prompt": prompt_f10,
        "thinking": thinking_f10,
        "code": code_f10,
    })

    return sft_rows, dpo_rows

base_sft_rows, base_dpo_rows = generate_all_task_families()
print(f"Generated {len(base_sft_rows)} unique base SFT templates and {len(base_dpo_rows)} unique base DPO templates.")


In [ ]:
# ==============================================================================
# CELL 5: LIVE AST & SANDBOX EXECUTION VERIFICATION OF GENERATED CODE TEMPLATES
# ==============================================================================
verified_count = 0
for idx, item in enumerate(base_sft_rows):
    # 1. Verify AST compilation
    ast.parse(item["code"])
    # 2. Execute representative scripts from each family in a subprocess against our 426-atom extxyz!
    if idx in (0, 3, 5, 6, 30, 33, 36, 38, 41):
        proc = subprocess.run(["python3", "-c", item["code"]], capture_output=True, text=True, timeout=30)
        assert proc.returncode == 0, f"Template {idx} ({item['family']}) failed: {proc.stderr}"
    verified_count += 1

print(f"100% of {verified_count} base templates passed AST compilation and live ASE subprocess execution!")


In [ ]:
# ==============================================================================
# CELL 6: FEATURE ENGINEERING & CURRICULUM AUGMENTATION
# ==============================================================================
def compute_code_features(row: Dict[str, Any]) -> Dict[str, Any]:
    code = row["code"]
    prompt = row["prompt"]
    thinking = row["thinking"]
    tree = ast.parse(code)

    # AST Node Count & Depth
    all_nodes = list(ast.walk(tree))
    ast_node_count = len(all_nodes)

    def get_depth(node):
        children = list(ast.iter_child_nodes(node))
        return 1 + max((get_depth(c) for c in children), default=0)
    ast_max_depth = get_depth(tree)

    # Cyclomatic Complexity (branches: For, While, If, IfExp, BoolOp, Try)
    branch_types = (ast.For, ast.While, ast.If, ast.IfExp, ast.BoolOp, ast.Try, ast.ListComp, ast.SetComp, ast.DictComp)
    cyclomatic_complexity = 1 + sum(isinstance(n, branch_types) for n in all_nodes)

    # Domain & ASE API Features
    ase_calls = len(re.findall(r"\b(read|write|neighbor_list|get_positions|get_chemical_symbols|get_all_distances|get_distance|set_pbc|wrap|FixAtoms|Langevin|MaxwellBoltzmannDistribution|fcc111|add_adsorbate)\b", code))
    uses_neighbor_list = int("neighbor_list(" in code)
    uses_pbc_mic = int("set_pbc(True)" in code or "mic=True" in code)
    has_phase_partitioning = int("water_set" in code or "o_w_" in code)
    has_headless_plot = int("plt.savefig(" in code and "plt.close()" in code)

    full_assistant = f"<thinking>\n{thinking}\n</thinking>\n\n```python\n{code}\n```"
    prompt_tokens_est = int(len((CANONICAL_SYSTEM_PROMPT + prompt).split()) * 1.35)
    completion_tokens_est = int(len(full_assistant.split()) * 1.35)
    total_tokens_est = prompt_tokens_est + completion_tokens_est

    # Curriculum Score: orders training from foundational single-concept scripts to complex multi-stage pipelines
    curriculum_score = round(
        row["difficulty_tier"] * 100.0 + 0.35 * ast_node_count + 1.5 * cyclomatic_complexity + 2.0 * ase_calls,
        2
    )

    # Exact Gemma Chat Template with explicit completion target
    formatted_chat = (
        f"<start_of_turn>user\n{CANONICAL_SYSTEM_PROMPT}\n\n{prompt}<end_of_turn>\n"
        f"<start_of_turn>model\n{full_assistant}<end_of_turn>"
    )

    return {
        **row,
        "assistant_response": full_assistant,
        "text": formatted_chat,
        "prompt_char_len": len(prompt),
        "code_char_len": len(code),
        "code_line_count": len(code.splitlines()),
        "prompt_tokens_est": prompt_tokens_est,
        "completion_tokens_est": completion_tokens_est,
        "total_tokens_est": total_tokens_est,
        "ast_node_count": ast_node_count,
        "ast_max_depth": ast_max_depth,
        "cyclomatic_complexity": cyclomatic_complexity,
        "ase_api_call_count": ase_calls,
        "uses_neighbor_list": uses_neighbor_list,
        "uses_pbc_mic": uses_pbc_mic,
        "has_phase_partitioning": has_phase_partitioning,
        "has_headless_plot": has_headless_plot,
        "curriculum_score": curriculum_score,
    }

# Expand with phrasing & cutoff perturbations to build a balanced ~630-example dataset
PROMPT_PREFIX_VARIATIONS = [
    "",
    "Output ONLY a single executable Python code block (no simulated console output). ",
    "Run inside the Docker sandbox container and return the result: ",
    "Using canonical ASE 3.29 and NumPy idioms, ",
    "For a headless container environment (Agg backend, 2GB RAM), ",
]

augmented_sft = []
for prefix in PROMPT_PREFIX_VARIATIONS:
    for row in base_sft_rows:
        mod_row = dict(row)
        mod_row["prompt"] = f"{prefix}{row['prompt']}".strip()
        augmented_sft.append(compute_code_features(mod_row))

# Sort by curriculum_score for Curriculum Learning in Stage 1 SFT
augmented_sft.sort(key=lambda r: r["curriculum_score"])
df_sft = pd.DataFrame(augmented_sft)
print(f"Engineered {len(df_sft)} SFT records with {len(df_sft.columns)} features.")
print(df_sft[["family", "difficulty_tier", "total_tokens_est", "ast_node_count", "cyclomatic_complexity", "curriculum_score"]].head(10))


In [ ]:
# ==============================================================================
# CELL 7: EXPLORATORY DATA ANALYSIS (EDA) & 4-PANEL PUBLICATION PLOT
# ==============================================================================
print("=== EDA SUMMARY BY TASK FAMILY ===")
family_summary = df_sft.groupby("family").agg(
    sample_count=("prompt", "count"),
    mean_tokens=("total_tokens_est", "mean"),
    mean_ast_nodes=("ast_node_count", "mean"),
    mean_cyclomatic=("cyclomatic_complexity", "mean"),
    mean_ase_calls=("ase_api_call_count", "mean"),
).reset_index()
print(family_summary.to_string(index=False))

fig, axs = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
((ax1, ax2), (ax3, ax4)) = axs

# Panel A: Samples per Task Family
fam_counts = df_sft["family"].value_counts()
ax1.barh(fam_counts.index, fam_counts.values, color="#2980b9", edgecolor="black")
ax1.set_xlabel("Number of Verified Samples")
ax1.set_title("A) Dataset Distribution Across Scientific Task Families", fontweight="bold")
ax1.grid(True, axis="x", linestyle="--", alpha=0.5)

# Panel B: Token Length Distribution vs 2048 Context Budget
ax2.hist(df_sft["prompt_tokens_est"], bins=15, alpha=0.7, color="#27ae60", label="Prompt Tokens")
ax2.hist(df_sft["completion_tokens_est"], bins=15, alpha=0.7, color="#e67e22", label="Completion (Thinking + Code) Tokens")
ax2.hist(df_sft["total_tokens_est"], bins=15, alpha=0.5, color="#8e44ad", label="Total Sequence Tokens")
ax2.axvline(2048, color="crimson", linestyle="--", lw=1.8, label="SFT Max Seq Length (2048)")
ax2.set_xlabel("Estimated Token Count")
ax2.set_ylabel("Frequency")
ax2.set_title("B) Sequence Token Length Distribution & Context Budget", fontweight="bold")
ax2.legend(loc="best")
ax2.grid(True, linestyle="--", alpha=0.5)

# Panel C: AST Node Count vs Cyclomatic Complexity
scatter = ax3.scatter(
    df_sft["ast_node_count"],
    df_sft["cyclomatic_complexity"],
    c=df_sft["ase_api_call_count"],
    cmap="viridis",
    s=60,
    edgecolors="black",
    alpha=0.8,
)
plt.colorbar(scatter, ax=ax3, label="ASE API Calls per Script")
ax3.set_xlabel("AST Node Count")
ax3.set_ylabel("Cyclomatic Complexity")
ax3.set_title("C) Code Complexity & ASE API Call Density", fontweight="bold")
ax3.grid(True, linestyle="--", alpha=0.5)

# Panel D: DPO Hard-Negative Error Taxonomy
dpo_expanded = []
for prefix in PROMPT_PREFIX_VARIATIONS:
    for r in base_dpo_rows:
        dpo_expanded.append({
            **r,
            "prompt": f"<start_of_turn>user\n{CANONICAL_SYSTEM_PROMPT}\n\n{prefix}{r['prompt']}<end_of_turn>\n<start_of_turn>model\n",
            "chosen": f"{r['chosen']}<end_of_turn>",
            "rejected": f"{r['rejected']}<end_of_turn>",
        })
df_dpo = pd.DataFrame(dpo_expanded)
err_counts = df_dpo["error_type"].value_counts()
ax4.bar(err_counts.index, err_counts.values, color="#c0392b", edgecolor="black")
ax4.set_ylabel("Preference Pair Count")
ax4.set_title("D) DPO Rejected Hard-Negative Defect Taxonomy", fontweight="bold")
ax4.tick_params(axis="x", rotation=18)
ax4.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
eda_plot_path = OUTPUT_DIR / "dataset_eda_summary.png"
plt.savefig(eda_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved 4-panel EDA figure to {eda_plot_path}")


In [ ]:
# ==============================================================================
# CELL 8: EXPORT EXACT JSONL & CSV FILES REQUIRED BY finetuning.ipynb
# ==============================================================================
# Stratified 90/10 Train/Validation split preserving curriculum order in training
val_indices = set(range(4, len(df_sft), 10))
sft_train = [r for i, r in enumerate(augmented_sft) if i not in val_indices]
sft_val = [r for i, r in enumerate(augmented_sft) if i in val_indices]

dpo_val_indices = set(range(4, len(dpo_expanded), 10))
dpo_train = [r for i, r in enumerate(dpo_expanded) if i not in dpo_val_indices]
dpo_val = [r for i, r in enumerate(dpo_expanded) if i in dpo_val_indices]

def write_jsonl(records: List[Dict[str, Any]], path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

write_jsonl(sft_train, OUTPUT_DIR / "sft_train.jsonl")
write_jsonl(sft_val, OUTPUT_DIR / "sft_val.jsonl")
write_jsonl(dpo_train, OUTPUT_DIR / "dpo_train.jsonl")
write_jsonl(dpo_val, OUTPUT_DIR / "dpo_val.jsonl")
df_sft.drop(columns=["code", "assistant_response", "text"]).to_csv(OUTPUT_DIR / "dataset_features_eda.csv", index=False)

manifest = {
    "dataset_name": "scicompute-gemma4-e2b-ase-md-dataset",
    "sft_train_count": len(sft_train),
    "sft_val_count": len(sft_val),
    "dpo_train_count": len(dpo_train),
    "dpo_val_count": len(dpo_val),
    "families": sorted(df_sft["family"].unique().tolist()),
    "files": [
        "sft_train.jsonl",
        "sft_val.jsonl",
        "dpo_train.jsonl",
        "dpo_val.jsonl",
        "dataset_features_eda.csv",
        "dataset_eda_summary.png",
        "ase_api_catalog.json",
        "Anatase_TiO2_Water_MolecularAdsorbedWater.extxyz",
    ]
}
with open(OUTPUT_DIR / "dataset_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("=== EXPORT COMPLETE ===")
print(json.dumps(manifest, indent=2))
